# RAG Fundamentals + Observable Evaluation in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Merged notebook:** RAG Fundamentals + Observable RAG Evaluation  
**Runtime target:** SageMaker Jupyter Notebook  
**Model access:** Amazon Bedrock  
**Data source:** S3  
**Supported input files:** `.docx`

This notebook combines the earlier **RAG Fundamentals** flow with the **Observable RAG Evaluation** flow into one classroom-ready lab.


## What learners will do

By the end of this notebook, learners will be able to:

- load `.docx`, `.txt`, and `.md` documents from S3
- compare chunking strategies and choose one for retrieval
- generate embeddings from a Bedrock embedding model available in the current region
- retrieve relevant chunks using in-memory similarity search
- optionally rerank retrieved chunks with an LLM
- generate grounded answers using Bedrock
- inspect retrieval traces, run metrics, and evaluation results in DataFrames
- improve the system during a mini-hack without saving any local artifacts



In [ ]:
# Uncomment only if your environment needs these libraries.
%pip install -q boto3 pandas numpy python-docx


## Step 1 — Configuration and imports

Update the S3 bucket and prefix before running the notebook.

**Instructor cue:**  
Explain that this notebook is intentionally simple enough for the classroom:
- Bedrock for model access
- S3 for source documents
- pandas DataFrames for traceability
- no local vector store


In [ ]:
from __future__ import annotations

import io
import json
import os
import re
from typing import Any

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from docx import Document as DocxDocument
from IPython.display import display


In [ ]:

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# LLM model for answer generation. Available models for this class
"""
amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
amazon.nova-pro-v1:0
"""
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"

# Embedding model. Available embedding model for this class
"""
amazon.titan-embed-text-v1 
amazon.titan-embed-text-v2:0
amazon.titan-embed-image-v1
"""
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

# S3 bucket
S3_BUCKET = "rag-demo-docs-sri"
S3_PREFIX = "rag-docs/"


ALLOWED_EXTENSIONS = (".docx")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 5
FINAL_K = 3

# Optional cap to make classroom runs faster. Set to None to embed every chunk.
MAX_CHUNKS_TO_EMBED = None

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
bedrock = boto3.client("bedrock", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("LLM model:", BEDROCK_LLM_MODEL_ID)
print("Requested embedding model:", BEDROCK_EMBEDDING_MODEL_ID or "AUTO-DISCOVER")
print("Allowed file types:", ALLOWED_EXTENSIONS)


## Step 3 — Load `.docx`, `.txt`, and `.md` documents from S3

This notebook reads directly from S3 and keeps the loaded corpus in memory.

**Discuss with learners:**
- production systems usually separate storage from compute
- S3 is the document source of truth here
- loaders are format-specific because `.docx` and `.txt` are not parsed the same way


In [ ]:
import io  # Used to handle the .docx file in memory as a byte stream
import pandas as pd  # Used to store the loaded document details in a DataFrame
from docx import Document as DocxDocument  # Used to read Word (.docx) files

# This function lists all .docx files inside a given S3 bucket and prefix(folder path)
def list_document_keys(bucket: str, prefix: str) -> list[str]:
    # Ask S3 to list objects inside the given bucket and prefix
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    # Create an empty list to store matching file keys
    keys = []

    # Loop through all objects returned by S3
    for obj in response.get("Contents", []):
        # Extract the file path/key from each object
        key = obj["Key"]
        
        # Keep only files that end with .docx
        if key.lower().endswith(".docx"):
            keys.append(key)

    # Return the sorted list of document keys
    return sorted(keys)

# This function reads one .docx file from S3 and converts it into plain text
def read_docx_from_s3(bucket: str, key: str) -> str:
    # Download the file from S3
    response = s3.get_object(Bucket=bucket, Key=key)
    
    # Read the file content as raw bytes
    raw = response["Body"].read()

    # Open the .docx file from memory using BytesIO
    doc = DocxDocument(io.BytesIO(raw))
    
    # Create an empty list to collect paragraph text
    parts = []

    # Loop through each paragraph in the Word document
    for p in doc.paragraphs:
        # Remove extra spaces from paragraph text
        text = p.text.strip()
        
        # Add only non-empty paragraphs
        if text:
            parts.append(text)

    # Join all paragraphs into one text block separated by blank lines
    return "\n\n".join(parts)

# Get the list of .docx files from the given S3 bucket and prefix
document_keys = list_document_keys(S3_BUCKET, S3_PREFIX)

# Print how many documents were found
print("Document count:", len(document_keys))

# Print the first 10 file keys so we can inspect them
print(document_keys[:10])



In [ ]:
# Create an empty list to store all loaded documents
raw_documents = []

# Loop through each document key
for key in document_keys:
    # Read the text content from the current .docx file
    text = read_docx_from_s3(S3_BUCKET, key).strip()

    # Add the document only if text is not empty
    if text:
        raw_documents.append({
            "source": key,             # Full S3 file path
            "file_type": "docx",       # File type is fixed as docx
            "text": text,              # Extracted document text
            "char_len": len(text),     # Number of characters in the text
        })

# Convert the list of dictionaries into a pandas DataFrame
docs_df = pd.DataFrame(raw_documents)

# Display the DataFrame in notebook output
display(docs_df)

# Print the final number of loaded documents
print("Loaded documents:", len(docs_df))

In [ ]:
# Student Activity 1: Inspect the loaded corpus
# Task:
# 1. How many documents are loaded?
# 2. Which document is the longest?
# 3. Which document is the shortest?

display(docs_df[["source", "char_len"]].sort_values("char_len", ascending=False))

## Step 4 — Compare chunking strategies

We compare three chunking approaches:

1. fixed-size chunking
2. fixed-size chunking with overlap
3. section-based chunking

**Instructor cue:**  
Use this section to help learners understand why chunking changes retrieval quality.


In [ ]:
# This function splits a long text into smaller pieces called chunks
def basic_chunking(text: str, chunk_size: int = 500) -> list[str]:
    # Create chunks of fixed size by moving from the start of the text
    # to the end in steps of chunk_size
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

# Apply the chunking function to the text of the first document in docs_df
basic_chunking(docs_df.iloc[0]["text"],10000)

In [ ]:
# This function splits text into chunks with overlap between consecutive chunks
def chunk_with_overlap(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    
    # Ensure overlap is smaller than chunk size to avoid infinite loop or invalid logic
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    
    # List to store all generated chunks
    chunks = []
    
    # Step size determines how much we move forward each time
    # Overlap means we move less than chunk_size
    step = chunk_size - overlap
    
    # Loop through the text using the step size
    for i in range(0, len(text), step):
        
        # Extract a chunk of size 'chunk_size' starting from index i
        chunk = text[i:i+chunk_size]
        
        # Add the chunk only if it is not empty
        if chunk:
            chunks.append(chunk)
    
    # Return the list of overlapping chunks
    return chunks

# Apply the function to the text of the first document
chunk_with_overlap(docs_df.iloc[0]["text"],100,5)


In [ ]:
import re  # Used for splitting the text based on blank lines

# This function splits the text into sections using blank lines as separators
def section_based_chunking(text: str, min_section_chars: int = 120) -> list[str]:
    
    # Split the text wherever there are blank lines, then remove extra spaces
    sections = [sec.strip() for sec in re.split(r"\n\s*\n", text) if sec.strip()]
    
    # Keep only sections that are at least the minimum required length
    return [sec for sec in sections if len(sec) >= min_section_chars]

# Check whether the DataFrame has any loaded documents
if docs_df.empty:
    raise ValueError("No documents loaded from S3.")

# Apply section-based chunking to the text of the first document
section_based_chunking(docs_df.iloc[0]["text"],200)



In [ ]:
# Get the source path of the first document from the DataFrame
sample_key = docs_df.iloc[0]["source"]

# Get the text content of the first document from the DataFrame
sample_text = docs_df.iloc[0]["text"]

# Apply all three chunking methods to the sample text
chunking_comparison = {
    "basic": basic_chunking(sample_text),                  # Fixed-size chunks
    "overlap": chunk_with_overlap(sample_text),           # Fixed-size chunks with overlap
    "section": section_based_chunking(sample_text),       # Section-based chunks using blank lines
}

# Create an empty list to store summary information for each chunking method
comparison_rows = []

# Loop through each chunking method and its generated chunks
for method, chunks in chunking_comparison.items():
    comparison_rows.append({
        "method": method,                                 # Name of the chunking method
        "n_chunks": len(chunks),                          # Total number of chunks created
        "first_chunk_preview": chunks[0][:300] if chunks else ""  # First 300 characters of the first chunk
    })

# Convert the summary list into a DataFrame
chunking_comparison_df = pd.DataFrame(comparison_rows)


In [ ]:
# Activity: Compare chunking strategies
# Task:
# Change CHUNK_SIZE and CHUNK_OVERLAP values mentally and discuss:
# 1. Which method creates more chunks?
# 2. Which method preserves more context?
# 3. Which method may increase cost?

display(chunking_comparison_df)

## Step 5 — Build chunk records with metadata

For the rest of the notebook we will use **overlap chunking**.  
Each chunk keeps metadata so that retrieval is explainable.

**Discuss with learners:**
- retrieval is better when chunks carry source metadata
- metadata helps filtering, debugging, and governance


In [ ]:
# Create an empty list to store all chunk-level rows
chunk_rows = []

# Loop through each document in docs_df
for row in docs_df.itertuples():
    
    # Apply overlapping chunking to the document text
    chunks = chunk_with_overlap(row.text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    
    # Loop through each chunk and also keep track of chunk number
    for i, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            "source": row.source,                              # Original document path/name
            "file_type": row.file_type,                        # File type of the document
            "chunk_id": f"{row.source}::chunk_{i:03d}",        # Unique chunk ID
            "chunk_index": i,                                  # Chunk number within the document
            "chunk_text": chunk,                               # Actual chunk text
            "char_len": len(chunk),                            # Number of characters in the chunk
        })

# Convert all chunk rows into a DataFrame
chunks_df = pd.DataFrame(chunk_rows)

# Display the first 10 chunk rows
display(chunks_df.head(10))

# Print total number of chunks created
print("Total chunk rows:", len(chunks_df))


## Step 6 — Embedding




In [ ]:
import json
import numpy as np

# Use the embedding model already defined earlier
EMBEDDING_MODEL_ID = BEDROCK_EMBEDDING_MODEL_ID

# This function calls Bedrock and returns embedding for given text
def get_text_embedding(text: str):
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text}),   # Titan expects inputText
        accept="application/json",
        contentType="application/json",
    )
    
    # Convert response to JSON and extract embedding vector
    return json.loads(response["body"].read())["embedding"]

# Generate embeddings for ALL chunks (no limit)
chunks_df["embedding"] = chunks_df["chunk_text"].apply(get_text_embedding)

# Store embedding size (dimension)
chunks_df["embedding_dim"] = chunks_df["embedding"].apply(len)

# Convert list of embeddings into a matrix (useful for similarity search later)
embedding_matrix = np.vstack(chunks_df["embedding"].values).astype("float32")

# Display sample rows (without full embedding vector)
display(chunks_df.drop(columns=["embedding"]).head(10))

# Print summary
print("Total chunks embedded:", len(chunks_df))
print("Embedding dimension:", chunks_df.iloc[0]["embedding_dim"])

## Step 7 — Retrieval by similarity


In [ ]:
import numpy as np
import pandas as pd

# Retrieve top_k relevant chunks for a query
def retrieve(query: str, top_k: int = TOP_K):

    # Convert query to embedding and normalize
    query_embedding = np.array(get_text_embedding(query), dtype="float32")
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Normalize all chunk embeddings and create matrix
    embedding_matrix = np.vstack([
        np.array(vec, dtype="float32") / np.linalg.norm(vec)
        for vec in chunks_df["embedding"]
    ])

    # Compute similarity scores
    scores = embedding_matrix @ query_embedding

    # Get top_k indices
    top_indices = np.argsort(-scores)[:top_k]

    # Collect results
    rows = []
    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        rows.append({
            "query": query,
            "rank": rank,
            "score": float(scores[idx]),
            "source": row["source"],
            "chunk_id": row["chunk_id"],
            "chunk_text": row["chunk_text"]
        })

    # Convert to DataFrame
    retrieval_df = pd.DataFrame(rows)

    # Build context string
    context = "\n\n".join(
        [f"[{r.rank}] {r.chunk_text}" for r in retrieval_df.itertuples()]
    )

    return retrieval_df, context


# Test
test_retrieval_df, test_context = retrieve(
    "What does the policy say about leave?",
    top_k=3
)

display(test_retrieval_df)
print(test_context[:1500])

In [ ]:
# Activity: Test retrieval quality
# Task:
# Try three different questions and check whether retrieved chunks are relevant.

query = "What does the policy say about leave?"

retrieval_df, context = retrieve(query, top_k=5)

display(retrieval_df[["rank", "score", "source", "chunk_text"]])

## Step 8 —Reranking with an LLM


In [ ]:
import re
import pandas as pd

# This function uses an LLM to rerank retrieved chunks
def rerank_results_llm(query: str, retrieval_df: pd.DataFrame, final_k: int = FINAL_K):
    
    # Step 1: Combine all retrieved chunks into a single prompt
    docs = ""
    for i, row in enumerate(retrieval_df.itertuples(index=False)):
        docs += f"\nDocument {i}:\n{row.chunk_text}\n"

    # Step 2: Create prompt for LLM
    prompt = f"""
You are a retrieval reranking assistant.

Given a query and multiple documents, return ONLY a Python list
of document numbers in order of relevance.

Query:
{query}

Documents:
{docs}

Example output:
[2, 0, 1]
"""

    # Step 3: Call LLM
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": 0}
    )

    # Step 4: Extract text response
    output = response["output"]["message"]["content"][0]["text"]

    # Step 5: Extract indices from response (e.g., [2,0,1])
    indices = list(map(int, re.findall(r"\d+", output)))

    # Step 6: Reorder rows based on LLM output
    reranked_rows = []
    for idx in indices[:final_k]:
        row = retrieval_df.iloc[idx]
        reranked_rows.append(row.to_dict())

    # Convert to DataFrame
    reranked_df = pd.DataFrame(reranked_rows)

    # Add rank position
    reranked_df["rerank_position"] = range(1, len(reranked_df) + 1)

    return reranked_df


# Step 7: Test retrieval + reranking
candidate_df, _ = retrieve(
    "What is eligibility for remote work?",
    top_k=TOP_K
)

reranked_df_preview = rerank_results_llm(
    "What is eligibility for remote work?",
    candidate_df,
    final_k=FINAL_K
)

# Display reranked results
display(reranked_df_preview)


In [ ]:
# Activity: Compare retrieval vs reranking
# Task:
# 1. Run retrieval
# 2. Run reranking
# 3. Check if reranking improved the order
# 4. Try with different queries and check if improvement is consistent
# 5. Change top_k and final_k values and check how it affects results

query = "What does the policy say about leave?"

retrieval_df, _ = retrieve(query, top_k=5)
reranked_df = rerank_results_llm(query, retrieval_df, final_k=3)

print("Original retrieval order:")
display(retrieval_df[["rank", "score", "source", "chunk_text"]])

print("Reranked order:")
display(reranked_df[["rerank_position", "score", "source", "chunk_text"]])

## Step 9 — Build grounded answers with Bedrock

This is the final RAG step:
1. retrieve
2. Rerank
3. build context
4. generate a grounded answer

**Discuss with learners:**
- the answer quality depends on the context quality
- RAG does not remove hallucination risk, but it reduces it when context is good


In [ ]:
import pandas as pd

# Call Bedrock LLM with a prompt and return the response text
def call_bedrock_llm(prompt: str):
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.2}
    )
    return response["output"]["message"]["content"][0]["text"]


# Build context string from retrieved chunks
def build_context_from_df(df: pd.DataFrame):
    return "\n\n".join(
        [f"[Source: {row.source}]\n{row.chunk_text}" for row in df.itertuples()]
    )


# Main RAG function
def answer_with_rag(query: str, retrieve_k: int = TOP_K, final_k: int = FINAL_K):
    
    # Step 1: Retrieve top chunks using embedding similarity
    retrieval_df, _ = retrieve(query, top_k=retrieve_k)
    
    # Step 2: Rerank using LLM (optional step simplified as always used)
    final_context_df = rerank_results_llm(query, retrieval_df, final_k=final_k)
    
    # Step 3: Build context from selected chunks
    context = build_context_from_df(final_context_df)
    
    # Step 4: Create final prompt for answer generation
    prompt = f"""
You are an enterprise RAG assistant.

Answer ONLY from the provided context.
If the answer is not present, say: "I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""
    
    # Step 5: Call LLM to generate final answer
    answer = call_bedrock_llm(prompt)
    
    # Step 6: Store basic run details
    run_metrics_df = pd.DataFrame([{
        "query": query,
        "retrieve_k": retrieve_k,
        "final_k": final_k,
        "n_documents": len(docs_df),
        "n_total_chunks": len(chunks_df),
        "timestamp_utc": pd.Timestamp.utcnow(),
    }])
    
    return {
        "retrieval_df": retrieval_df,          # Initial retrieved chunks
        "final_context_df": final_context_df,  # After reranking
        "answer": answer,                      # Final LLM answer
        "run_metrics_df": run_metrics_df,      # Simple run metrics
    }


# Run a demo query
demo_run = answer_with_rag("What does the policy say about leave?")

# Print final answer
print("ANSWER\n")
print(demo_run["answer"])

# Show retrieved chunks
print("\nInitial retrieval")
display(demo_run["retrieval_df"])

# Show reranked chunks
print("\nFinal context after reranking")
display(demo_run["final_context_df"])

# Show run metrics
print("\nRun metrics")
display(demo_run["run_metrics_df"])

In [ ]:
# Activity: Grounded vs ungrounded question
# Task:
# Ask one question that is answerable from the documents.
# Ask one question that is NOT answerable from the documents.

answerable_question = "What does the policy say about leave?"
unanswerable_question = "Who won yesterday's IPL match?"

print("Answerable question:")
print(answer_with_rag(answerable_question)["answer"])

print("\nUnanswerable question:")
print(answer_with_rag(unanswerable_question)["answer"])

# RAG pipeline step-wise breakdown

| Stage | What happens | Model Type | # Calls | Token Usage | Cost Driver | When it runs |
|------|-------------|------------|--------|-------------|-------------|--------------|
| Data Ingestion |  |  |  |  |  |  |
| Text Extraction |  |  |  |  |  |  |
| Chunking |  |  |  |  |  |  |
| Embedding Generation |  |  |  |  |  |  |
| Storage (Vector DB) |  |  |  |  |  |  |
| Query Embedding |  |  |  |  |  |  |
| Retrieval (Similarity Search) |  |  |  |  |  |  |
| Reranking (optional) |  |  |  |  |  |  |
| Context Building |  |  |  |  |  |  |
| Final Answer Generation |  |  |  |  |  |  |

# RAG Evaluation

In this section, we evaluate the RAG pipeline using metrics that work reliably in this notebook environment.

We will calculate:

- **Exact Match**: checks whether the predicted answer exactly matches the reference answer
- **ROUGE**: checks text overlap between predicted answer and reference answer


These metrics help us evaluate both:
- **answer quality**
- **retrieval quality**



## ROUGE Score (Evaluation of Generated Text)

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a metric used to compare a **generated answer** with a **reference (ground truth) answer**.

It measures how much overlap exists between the two texts.

### Types of ROUGE used

- **ROUGE-1**  
  Measures overlap of individual words (unigrams)  
  → “How many words match?”

- **ROUGE-2**  
  Measures overlap of word pairs (bigrams)  
  → “How many word sequences match?”

- **ROUGE-L**  
  Measures longest common sequence of words  
  → “How similar is the overall structure?”

### What does the score mean?

ROUGE values range from **0 to 1**:
- **1.0** → perfect match
- **0.0** → no overlap

Higher ROUGE score means the generated answer is closer to the reference answer.

### Important Note

ROUGE checks **text similarity**, not correctness:
- A high score means answers look similar
- It does NOT guarantee the answer is factually correct

---

## Why Accuracy, Precision, and F1 Score are NOT used here

Accuracy, Precision, and F1 Score are designed for **classification tasks**, where outputs are from a fixed set of labels.

Examples:
- Yes / No  
- Category A / B / C  
- Relevant / Not Relevant  

In this notebook, the model generates **free-text answers**, not labels.

Example:
- Ground truth: "Employees are eligible after 6 months"
- Model output: "Eligibility starts after six months of employment"

These are **correct but not exact matches**, so:
- Accuracy → would mark this as wrong ❌  
- F1 / Precision → not meaningful for sentence-level text  

---

## Why ROUGE is suitable for RAG

ROUGE works better because:
- It measures **partial overlap**
- It captures **similar meaning expressed differently**
- It is designed for **text generation tasks**

---

## Summary

- Use **Accuracy / F1 / Precision** → for classification problems  
- Use **ROUGE** → for text generation problems (like RAG)  

In [ ]:
%pip install rouge_score -q
import io
import pandas as pd
import boto3
from rouge_score import rouge_scorer

# Set S3 details
S3_BUCKET = "rag-demo-docs-sri"
S3_KEY = "rag_evaluation_qna_15.csv"

# Create S3 client and read file
s3 = boto3.client("s3")
response = s3.get_object(Bucket=S3_BUCKET, Key=S3_KEY)
eval_df = pd.read_csv(io.BytesIO(response["Body"].read()))

# Show the loaded file
display(eval_df.head())
print("Shape:", eval_df.shape)
print("Columns:", eval_df.columns.tolist())

### Run RAG for Each Question

In [ ]:
# Store model answers and retrieved contexts
predicted_answers = []


# Run the RAG pipeline for each question
for question in eval_df["question"]:
    result = answer_with_rag(question)
    predicted_answers.append(result["answer"])


# Add model outputs to the evaluation DataFrame
eval_df["predicted_answer"] = predicted_answers


# Show sample output
display(eval_df.head())

### Exact Match

In [ ]:
# Normalize text before comparison
def normalize_text(text):
    return str(text).strip().lower()

# Exact Match = 1 if answer and predicted answer are exactly the same after normalization
eval_df["exact_match"] = (
    eval_df["answer"].apply(normalize_text) ==
    eval_df["predicted_answer"].apply(normalize_text)
).astype(int)

print("Exact Match Score:", round(eval_df["exact_match"].mean(), 4))

### ROUGE Scores

In [ ]:
# Create ROUGE scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# Store ROUGE scores for each row
rouge_rows = []

for row in eval_df.itertuples(index=False):
    scores = scorer.score(str(row.answer), str(row.predicted_answer))
    
    rouge_rows.append({
        "question": row.question,
        "rouge1_fmeasure": scores["rouge1"].fmeasure,
        "rouge2_fmeasure": scores["rouge2"].fmeasure,
        "rougeL_fmeasure": scores["rougeL"].fmeasure,
    })

# Convert to DataFrame
rouge_df = pd.DataFrame(rouge_rows)

# Show sample scores
display(rouge_df.head())

# Print average ROUGE scores
print("Average ROUGE-1:", round(rouge_df["rouge1_fmeasure"].mean(), 4))
print("Average ROUGE-2:", round(rouge_df["rouge2_fmeasure"].mean(), 4))
print("Average ROUGE-L:", round(rouge_df["rougeL_fmeasure"].mean(), 4))

### Context Hit Rate and Final Summary

In [ ]:

# Final summary table
summary_df = pd.DataFrame([{
    "exact_match": round(eval_df["exact_match"].mean(), 4),
    "avg_rouge1": round(rouge_df["rouge1_fmeasure"].mean(), 4),
    "avg_rouge2": round(rouge_df["rouge2_fmeasure"].mean(), 4),
    "avg_rougeL": round(rouge_df["rougeL_fmeasure"].mean(), 4),
}])

display(summary_df)